In [1]:
import importlib
import csv
import os

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from config import config2
from models.backbone import get_backbone
from datasets.data import MyData
from models.patchcore import PatchCore
from utils.metrics import get_image_auc #get_pixel_auc
from utils.visualization import denormalize, show_result

print("class:", config2.CLASS_NAME)
print("k:", config2.K)
print("train limit:", config2.TRAIN_LIMIT)
print("test limit:", config2.TEST_LIMIT)
print("test limit per class:", config2.TEST_LIMIT_PER_CLASS)

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(device)

classn = ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut'
          ,'leather','metal_nut','pill','screw','tile','toothbrush','transistor'
          ,'wood','zipper']

for name in classn:
    train_data = MyData(
    name,
    phase="train",
    batch_size=config2.BATCH_SIZE,
    shuffle=False,
    limit=config2.TRAIN_LIMIT,
    )

    test_data = MyData(
        name,
        phase="test",
        batch_size=config2.BATCH_SIZE,
        shuffle=False,
        limit=config2.TEST_LIMIT,
        limit_per_class=config2.TEST_LIMIT_PER_CLASS,
    )
    backbone = get_backbone()
    patchcore = PatchCore(backbone, k=config2.K, device=device)

    patchcore.fit(train_data)

    IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


    #show_result(config.TEST_INDEX)
    scores = []

    for i in range(len(test_data)):
        img, label = test_data[i]
        score, _ = patchcore.predict(img)
        scores.append((i, score.item(), test_data.get_classes()[label]))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores[:10]
    image_auc = get_image_auc(test_data, patchcore)
    image_auc

    file_path = "results.csv"
    file_exists = os.path.isfile(file_path)

    with open(file_path, 'a', newline="") as f:
        writer = csv.writer(f)

        if not file_exists:
            writer.writerow(["category","train_limit","k","auc","anomaly_score"])
        writer.writerow([name, config2.TRAIN_LIMIT, config2.K, image_auc,score ])
            


class: zipper
k: 50
train limit: 50
test limit: 30
test limit per class: 3
mps


In [2]:
import pandas as pd